# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

march_path = hf_hub_download(
      repo_id="FlyRank/internship-warehouse",
      filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
      repo_type="dataset",
      token=hf_token
  )

df = pd.read_parquet(march_path)

print("rows loaded:", len(df))
print("\ncolumns:")
print(df.columns.tolist())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

rows loaded: 9841378

columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [2]:
key_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

avaliable = [c for c in key_fields if c in df.columns]

print("Ket fields found:", avaliable)
print("\nDistribution summary:")
print(
    df[avaliable]
    .describe(percentiles=[0.50,  0.75, 0.90, 0.95, 0.99])
    .T
)

Ket fields found: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

Distribution summary:
                      count       mean         std  min  50%   75%   90%  \
gsc_impressions   9841378.0  28.518119  155.926569  0.0  0.0   6.0  54.0   
gsc_clicks        9841378.0   0.083508    0.781434  0.0  0.0   0.0   0.0   
gsc_avg_position  3611061.0  15.826651   19.856034  0.0  7.5  20.2  43.0   

                     95%     99%      max  
gsc_impressions   135.00  509.00  40084.0  
gsc_clicks          0.00    2.00    274.0  
gsc_avg_position   62.75   88.75    498.0  


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal #1__ Search position and CTR
Claim: Pages with better search positions tend to have a higher CTR
Test: I grouped pages by search position and compared total clicks / total impressions.
Verdict: CONFIRMED. CTR was highest for positions 1-3 and generally decreased for weaker positions.

In [3]:
test1 = df[
    (df["gsc_impressions"] > 0) &
    (df["gsc_avg_position"].notna())
].copy()

test1["ctr"] = test1["gsc_clicks"] / test1["gsc_impressions"]

test1["position_group"] = pd.cut(
    test1["gsc_avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)

result1 = test1.groupby("position_group", observed=True).agg(
    rows=("ctr", "size"),
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum")
)

result1["ctr"] = result1["clicks"] / result1["impressions"]

print(result1)




                   rows  impressions  clicks       ctr
position_group                                        
1-3              564173     53559848  204283  0.003814
4-10            1456122    137830113  445828  0.003235
11-20            519223     29386006   92449  0.003146
21+              908354     59412876   78098  0.001314


Signal #2__Impressions and clicks
Claim: Pages with more search impressions tend to receive more clicks.
Test: I used Spearman correlation because the data is heavy-tailed.
Verdict: CONFIRMED. impressions and clicks had a positive spearman correlation of 0.4187

In [6]:
test2 = df[df["gsc_impressions"] > 0]

correltaion = test2[
    ["gsc_impressions", "gsc_clicks"]
].corr(method="spearman")

print(correltaion)

                 gsc_impressions  gsc_clicks
gsc_impressions           1.0000      0.4187
gsc_clicks                0.4187      1.0000


Signal#3__ Search position and impressions
Claim: Pages with better search positions tend to receive more impressions.
Test: I used Spearman correlation because the data is heavy tailed.
Verdict: MIXED. The correlation was negative(-0.0801) which is in the expected direction but the relationship was very weak.

In [7]:
test3 = df[
    (df["gsc_impressions"] > 0)&
    (df["gsc_avg_position"].notna())
]

correlation3 = test3[
    ["gsc_avg_position", "gsc_impressions"]
].corr(method="spearman")

print(correlation3)

                  gsc_avg_position  gsc_impressions
gsc_avg_position           1.00000         -0.08013
gsc_impressions           -0.08013          1.00000


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag_linked test _ VISIBLE_LOW_CTR
Claim: The VISIBLE_LOW_CTR flag assumes that pages with weaker search positions tend to have lower CTR,
Test: I grouped pages with impressions by search position and compared total clicks/ total impressions.
Verdict: MIXED. CTR was much lower for position 21+ but it did not decrease consistently across every position group. The data supports the general idea behind the flag but not as a strict rule.

In [11]:
flag_test = df[
    (df["gsc_impressions"] > 50)&
               (df["gsc_avg_position"].notna())
].copy()

flag_test["ctr"] = flag_test["gsc_clicks"] / flag_test["gsc_impressions"]

print("Rows tested:", len(flag_test))
print("Median CTR:", flag_test["ctr"].median())

flag_test["position_group"] = pd.cut(
    flag_test["gsc_avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)

flag_result = flag_test.groupby("position_group", observed=True).agg(
    rows=("ctr", "size"),
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum")
)

flag_result["ctr"] = flag_result["clicks"] / flag_result["impressions"]

print(flag_result)

Rows tested: 1025031
Median CTR: 0.0
                  rows  impressions  clicks       ctr
position_group                                       
1-3             194865     48219489  185261  0.003842
4-10            502071    124580526  402136  0.003228
11-20           136015     23302130   77951  0.003345
21+             191836     52002505   69254  0.001332


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team can use low CTR as a signal for pages that may need review especially when they still receive meaningful impressions. However search position also matters so low CTR should not be treated as proof that a page has conntent problem. The flag should support human review rather than trigger automatic changes.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.